# 01 — Data Quality: What Was Removed, and Why

UCI "Online Retail II" (id 502, CC BY 4.0), ~1.07M raw transaction line items. Every cleaning
rule is explicit SQL in `../sql/00_clean.sql`, run through DuckDB — nothing is silently dropped
in pandas. See `../data/README.md` for the source and `../.ai/PROJECT_SPEC.md` for the brief.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..", "src").resolve()))

from retail_analytics.engine import build_connection

con = build_connection()
n_raw = con.sql("SELECT COUNT(*) AS n FROM transactions_raw").df()["n"][0]
print(f"Raw rows: {n_raw:,}")

Raw rows: 1,067,371


## What was removed, by rule

In [2]:
removed = con.sql("SELECT * FROM removed_rows_summary").df()
removed

,removal_reason,n_rows,pct_of_total
0,kept,802634,75.20
1,missing_customer_id,240459,22.53
2,cancelled_invoice,19494,1.83
3,non_product_stock_code,4723,0.44
4,non_positive_quantity_or_price,61,0.01


**Rule 1 — cancelled invoices** (`Invoice` starts with `'C'`): this dataset records a
return/cancellation as a brand-new invoice, not a signed reversal row on the original — these
represent stock going back, not a completed sale.

**Rule 2 — non-product stock codes**: postage, bank fees, manual/test entries, carriage
adjustments, and gift-card denominations share the `StockCode` column with real products but
aren't merchandise sales. The blocklist was built by manually inspecting all 68 distinct
non-numeric-pattern stock codes — genuine-if-oddly-formatted products (e.g. the `DCGS####`
Dotcom Gift Shop line, colour-suffixed codes like `15056BL`) were deliberately kept, not
excluded, since they *are* real merchandise.

**Rule 3 — missing customer ID**: roughly a quarter of rows have no `Customer ID` (guest /
unregistered checkouts). Every downstream table in this project — RFM, cohorts, revenue
concentration — is customer-level, so these rows can't be attributed to anyone and are excluded.

**Rule 4 — non-positive quantity or price**: a small number of residual manual/adjustment rows
not already caught by rules 1-3 — not a real product sale at a real price.

In [3]:
total_removed = removed.loc[removed["removal_reason"] != "kept", "n_rows"].sum()
total_kept = removed.loc[removed["removal_reason"] == "kept", "n_rows"].iloc[0]
print(f"Kept: {total_kept:,} rows ({100 * total_kept / n_raw:.1f}%)")
print(f"Removed: {total_removed:,} rows ({100 * total_removed / n_raw:.1f}%)")

Kept: 802,634 rows (75.2%)
Removed: 264,737 rows (24.8%)


## Sanity checks on the cleaned data

In [4]:
checks = con.sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT customer_id) AS n_customers,
        COUNT(DISTINCT invoice) AS n_invoices,
        MIN(invoice_date) AS earliest_date,
        MAX(invoice_date) AS latest_date,
        ROUND(SUM(revenue), 2) AS total_revenue,
        SUM(CASE WHEN revenue < 0 THEN 1 ELSE 0 END) AS negative_revenue_rows,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_ids
    FROM transactions_clean
""").df()
checks

,n_rows,n_customers,n_invoices,earliest_date,latest_date,total_revenue,negative_revenue_rows,null_customer_ids
0,802634,5852,36594,2009-12-01 07:45:00,2011-12-09 12:50:00,17434464.73,0.0,0.0


No negative revenue and no null customer IDs remain — both cleaning rules did what they were
meant to do. `customer_base`, `rfm_scores`, `cohort_retention`, and `revenue_concentration`
(built in `notebooks/02_customer_base.ipynb`) all read from `transactions_clean`, so every number
downstream inherits this same, documented cleaning.